In [1]:
import sys
import os

# Jupyter automatically adds the notebook's directory to sys.path
# (that's why '/Users/dorian/ML4Finance/Project-C-Brochud/Project/src/models/hybrid_system' is in sys.path)
# We need to add the project root so 'from src.data...' imports work
project_root = '/Users/dorian/ML4Finance/Project-C-Brochud/Project'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from src.data.process_tech_text_ds import make_fusion_dataset


/opt/miniconda3/envs/advancedMLToolKit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.optim as optim
import torch.nn as nn
from configs.config import FUSION_DATA_DIR
from datasets import load_from_disk
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [18]:
import torch
import torch.nn as nn

class TextAttentionNet(nn.Module):
    def __init__(self, embed_dim=768, hidden_dim=64):
        super(TextAttentionNet, self).__init__()
        # 1. Per-Headline Processor
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, hidden_dim),
            nn.Tanh()
        )
        
        # 2. Attention Aggregator (Many headlines -> One vector)
        self.attention = nn.Linear(hidden_dim, 1)
        
        # 3. Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (Batch, Max_Headlines, Embed_Dim)
        
        # Pass features: (Batch, Max_H, Hidden)
        features = self.encoder(x)
        
        # Calculate Attention Scores
        # attn_weights: (Batch, Max_H, 1)
        attn_weights = torch.softmax(self.attention(features), dim=1)
        
        # Weighted Sum: (Batch, Hidden)
        context_vector = torch.sum(features * attn_weights, dim=1)
        
        # Final Probability
        return self.classifier(context_vector)

In [19]:
# trainer for torch
def train_pytorch(model, train_loader, val_loader, epochs=100,):
    """Internal helper to train the neural net loop"""
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    
    
    for epoch in range(epochs):
        total_loss = 0
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to("cpu"), y_batch.to("cpu")
            
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
        epoch_loss = total_loss / len(train_loader) 

        # validation 
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to("cpu"), labels.to("cpu")
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * inputs.size(0)
        epoch_val_loss = running_val_loss / len(val_loader)

        print(f"  Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

In [20]:
data_text = load_from_disk(os.path.join(FUSION_DATA_DIR, "data_text.arrow"))
data_price = pd.read_csv(os.path.join(FUSION_DATA_DIR, "data_price.csv"))

In [21]:
X, y = data_text["embedding"], data_text["Label"]


In [22]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32), 
        torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    )
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


val_dataset = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32), 
        torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    )
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [24]:
model = TextAttentionNet()

In [25]:
train_pytorch(model, train_loader, val_loader)

  Epoch 1/100 | Train Loss: 11.0592 | Val Loss: 10.9501
  Epoch 2/100 | Train Loss: 11.0476 | Val Loss: 10.8129
  Epoch 3/100 | Train Loss: 11.0348 | Val Loss: 10.8139
  Epoch 4/100 | Train Loss: 11.0238 | Val Loss: 10.8551
  Epoch 5/100 | Train Loss: 11.0120 | Val Loss: 10.8249
  Epoch 6/100 | Train Loss: 11.0165 | Val Loss: 10.8676
  Epoch 7/100 | Train Loss: 11.0019 | Val Loss: 10.8284
  Epoch 8/100 | Train Loss: 10.9754 | Val Loss: 10.9361
  Epoch 9/100 | Train Loss: 11.0315 | Val Loss: 10.8302
  Epoch 10/100 | Train Loss: 10.9872 | Val Loss: 10.8543
  Epoch 11/100 | Train Loss: 10.9818 | Val Loss: 10.8499
  Epoch 12/100 | Train Loss: 10.9206 | Val Loss: 11.0467
  Epoch 13/100 | Train Loss: 10.8398 | Val Loss: 10.9210
  Epoch 14/100 | Train Loss: 10.5921 | Val Loss: 11.2026
  Epoch 15/100 | Train Loss: 10.3076 | Val Loss: 11.6405
  Epoch 16/100 | Train Loss: 10.2813 | Val Loss: 11.1605
  Epoch 17/100 | Train Loss: 9.7201 | Val Loss: 12.0619
  Epoch 18/100 | Train Loss: 9.0715 | Val